# Position probe inspection

Single-image inference + attention visualization for the **position-between-objects** probe.

- Loads a frozen backbone, loads a trained probe head, and runs a forward pass on one image.
- Shows the predicted direction (Front / Back / Left / Right) vs. the ground truth computed
  from the scene's `params_*.json`, with an attention overlay for the attention-based probes
  (`cls_efficient` / `cls_abmilp`).

**Prerequisites**
- Install the package: `pip install -e .` from the repo root.
- This example uses the **VGGT** backbone, so set `VGGT_REPO` to your local
  [VGGT](https://github.com/facebookresearch/vggt) checkout.
- Bundled assets (committed to the repo): example images in `notebook/examples/` and a matching
  trained head (`vggt_l16` / `cls_efficient`, `winter_town_2`, Snowman&rarr;Husky) in
  `notebook/heads/`. Run this notebook from the `notebook/` directory so the relative paths
  resolve, or override them with the env vars below.

In [ ]:
import os
from pathlib import Path

# ---- Paths (override via env vars) ----
# Bundled example images + their params_*.json (run from the notebook/ directory).
EXAMPLES_DIR = Path(os.environ.get("SPARRTA_NOTEBOOK_EXAMPLES", "examples"))
# Trained probe-head checkpoints. The repo bundles one under notebook/heads/.
HEAD_ROOT = Path(os.environ.get("SPARRTA_HEADS_DIR", "heads"))

# ---- User parameters ----
ENVIRONMENT = "winter_town_2"
IMAGE_PATH = EXAMPLES_DIR / "img_0015.jpg"
MODEL_NAME = "vggt_l16"          # backbone key (this notebook wires up vggt_l16)
PROBE_NAME = "cls_efficient"     # cls_linear, cls_abmilp, cls_efficient
PERSPECTIVE = "camera"           # camera or human
REFERENCE_LABEL = "Snowman"
TARGET_LABEL = "Husky"
DEVICE_PREFERENCE = "cuda"       # "cuda" or "cpu"
IMAGE_MEAN = "imagenet"          # or "clip"
IMAGE_SIZE = 224
AMBIGUITY_DEGREES = 10           # 10 or 15
FRONT_DEGREES = 45
BACK_DEGREES = 135

In [ ]:
import torch
import numpy as np
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from typing import Optional

from sparrta.datasets.unreal_position import (
    LABEL_TO_INDEX,
    INDEX_TO_LABEL,
    _matching_json_for_image,
    _load_positions,
    classify_relative_direction,
)
from sparrta.models.vggt import VGGT1B
from sparrta.models.probes import ClassificationHead, EfficientProbing, ABMILPHead

device = torch.device(
    DEVICE_PREFERENCE if DEVICE_PREFERENCE == "cpu" or torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")

# Map the probe choice to the backbone pooling flags it expects.
if PROBE_NAME == "cls_linear":
    return_cls, mean_pool, efficient_probe = False, True, False
elif PROBE_NAME in ("cls_abmilp", "cls_efficient"):
    return_cls, mean_pool, efficient_probe = False, False, True
else:
    raise ValueError(f"Unsupported probe: {PROBE_NAME}")

In [ ]:
def get_mean_std(image_mean):
    if isinstance(image_mean, (list, tuple)):
        return [float(m) for m in image_mean], [1.0, 1.0, 1.0]
    if image_mean == "imagenet":
        return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if image_mean == "clip":
        return [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]
    return [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]


def load_image_and_label(image_path, perspective, reference_label, target_label,
                         image_size, image_mean, ambiguity_degrees, front_degrees, back_degrees):
    mean, std = get_mean_std(image_mean)
    transform = T.Compose([
        T.Resize((image_size, image_size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    with Image.open(image_path).convert("RGB") as im:
        pil_img = im.copy()
    tensor_img = transform(pil_img)

    json_path = _matching_json_for_image(Path(image_path))
    label_idx, label_name = None, None
    if json_path is not None:
        positions = _load_positions(json_path, reference_label, target_label, human_label="Human")
        if positions is not None:
            observer = positions.get("camera" if perspective.lower() == "camera" else "human")
            if observer is not None:
                label_name = classify_relative_direction(
                    observer, positions["reference"], positions["target"],
                    ambiguity_degrees, front_degrees, back_degrees,
                )
                label_idx = LABEL_TO_INDEX.get(label_name)
    return tensor_img, label_idx, label_name, mean, std


def load_backbone(model_name):
    if model_name == "vggt_l16":
        repo_dir = os.environ.get("VGGT_REPO", "")
        if not repo_dir:
            raise EnvironmentError(
                "The vggt_l16 backbone needs the external VGGT repository. Set VGGT_REPO to a "
                "local clone of https://github.com/facebookresearch/vggt."
            )
        model = VGGT1B(repo_dir=repo_dir, add_norm=True, return_cls=return_cls,
                       mean_pool=mean_pool, efficient_probe=efficient_probe)
    else:
        raise NotImplementedError(f"Backbone {model_name} is not wired up in this notebook.")
    model.eval().to(device)
    return model


def build_head(probe_name, feat_dim, num_classes=4):
    probe_name = probe_name.lower()
    if probe_name == "cls_linear":
        head = ClassificationHead(feat_dim=feat_dim, num_classes=num_classes, use_layernorm=True)
    elif probe_name == "cls_efficient":
        head = EfficientProbing(feat_dim=feat_dim, num_classes=num_classes, use_layernorm=True)
    elif probe_name == "cls_abmilp":
        head = ABMILPHead(feat_dim=feat_dim, num_classes=num_classes, use_layernorm=True)
    else:
        raise ValueError(f"Unsupported probe: {probe_name}")
    head.eval().to(device)
    return head


def find_head_checkpoint(root, model_name, probe_name, environment, perspective,
                         reference_label, target_label):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Head root {root} does not exist")
    tokens = [model_name, probe_name, environment, perspective, reference_label, target_label]
    candidates = []
    for pt in root.rglob("*.pt"):
        path_str = pt.as_posix().lower().replace(" ", "-")
        if all(tok.lower().replace(" ", "-") in path_str for tok in tokens):
            candidates.append(pt)
    if not candidates:
        raise FileNotFoundError(f"No head checkpoint in {root} matching tokens {tokens}")
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def denorm_image(tensor_img, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    img = (tensor_img.cpu() * std + mean).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()


def overlay_attention(image_np, attention_map, head, backbone):
    if attention_map is None:
        return image_np
    attn = attention_map.detach().cpu()
    if attn.ndim == 3:
        attn = attn.squeeze(0)
    gh = image_np.shape[0] // backbone.patch_size
    gw = image_np.shape[1] // backbone.patch_size
    try:
        attn = attn.reshape(head.num_queries, gh, gw)
    except Exception:
        attn = attn[:, 1:].reshape(head.num_queries, gh, gw)
    attn = attn.mean(dim=0, keepdim=True).unsqueeze(0)
    attn = torch.nn.functional.interpolate(
        attn, scale_factor=(backbone.patch_size, backbone.patch_size), mode="nearest"
    )[0].permute(1, 2, 0).numpy()
    attn = cv2.blur(attn, (8, 8))
    attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    heatmap = plt.get_cmap("jet")(attn)[:, :, :3]
    alpha = 0.25
    return (1 - alpha) * image_np + alpha * heatmap

## Single image

Load the backbone + trained head once, then run inference on `IMAGE_PATH`.

In [ ]:
# Load the backbone and the trained head once.
backbone = load_backbone(MODEL_NAME)

# Probe the feature dimension with a dummy forward, then build + load the head.
image_tensor, gt_idx, gt_label, mean, std = load_image_and_label(
    IMAGE_PATH, PERSPECTIVE, REFERENCE_LABEL, TARGET_LABEL,
    IMAGE_SIZE, IMAGE_MEAN, AMBIGUITY_DEGREES, FRONT_DEGREES, BACK_DEGREES,
)
feat = backbone(image_tensor.unsqueeze(0).to(device))
feat_dim = feat.shape[-1]
print(f"Feature shape: {tuple(feat.shape)}")

head_ckpt = find_head_checkpoint(
    HEAD_ROOT, MODEL_NAME, PROBE_NAME, ENVIRONMENT, PERSPECTIVE, REFERENCE_LABEL, TARGET_LABEL,
)
head = build_head(PROBE_NAME, feat_dim, num_classes=4)
head.load_state_dict(torch.load(head_ckpt, map_location=device))
head.eval().to(device)
print(f"Loaded head from {head_ckpt}")


def predict(image_path):
    image_tensor, gt_idx, gt_label, mean, std = load_image_and_label(
        image_path, PERSPECTIVE, REFERENCE_LABEL, TARGET_LABEL,
        IMAGE_SIZE, IMAGE_MEAN, AMBIGUITY_DEGREES, FRONT_DEGREES, BACK_DEGREES,
    )
    with torch.no_grad():
        feat = backbone(image_tensor.unsqueeze(0).to(device))
        logits = head(feat)
        probs = torch.softmax(logits, dim=1)
        pred_idx = int(probs.argmax(dim=1).item())
    pred_label = INDEX_TO_LABEL.get(pred_idx, str(pred_idx))
    attn_map = head.attention_map if hasattr(head, "attention_map") else None
    return image_tensor, gt_label, pred_label, float(probs[0, pred_idx]), attn_map, mean, std


image_tensor, gt_label, pred_label, prob, attn_map, mean, std = predict(IMAGE_PATH)
print(f"GT: {gt_label} | Pred: {pred_label} (p={prob:.3f})")

In [ ]:
# Visualization: image with an attention overlay for the attention-based probes.
image_np = denorm_image(image_tensor, mean, std)
if PROBE_NAME.lower() == "cls_linear":
    blended = image_np
else:
    blended = overlay_attention(image_np, attn_map, head, backbone)

plt.figure(figsize=(4, 4))
plt.imshow(blended)
plt.axis("off")
plt.title(f"GT: {gt_label} | Pred: {pred_label}")
plt.show()

## All bundled examples

Run the same probe over every bundled example (one per direction: Right, Left, Back, Front).

In [ ]:
IMAGES = [EXAMPLES_DIR / f"img_{i:04d}.jpg" for i in (15, 16, 17, 21)]

fig, axes = plt.subplots(1, len(IMAGES), figsize=(4 * len(IMAGES), 4))
for ax, image_path in zip(np.atleast_1d(axes), IMAGES):
    image_tensor, gt_label, pred_label, prob, attn_map, mean, std = predict(image_path)
    image_np = denorm_image(image_tensor, mean, std)
    blended = image_np if PROBE_NAME.lower() == "cls_linear" else overlay_attention(
        image_np, attn_map, head, backbone)
    ax.imshow(blended)
    ax.axis("off")
    ax.set_title(f"GT: {gt_label}\nPred: {pred_label} ({prob:.2f})")
plt.tight_layout()
plt.show()